In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
customer = spark.read.table("fabric_lh.bronze.customers")


In [0]:
display(customer)

customer_id,name,EMAIL,gender,dob,location
1001,john DOE,John.DOE@Gmail.Com,male,1990/05/12,new york
1002,sara kHAN,SARA.khan@GMAIL.com,F,12-06-1988,MUMBAI
1003,null,tom@outlook.com,Male,null,san francisco
1004,Anil Sharma,ANIL.sharma@yahoo.in,MALE,1989-03-17,bangalore
1005,Priya R,priya.r@hotmail.com,f,1993-11-23,delhi
1006,Amit Patel,amit123@gmail.com,M,1992/08/19,surat
1007,Meena Kumari,meena@rediffmail.com,female,1995-04-10,chennai
1008,Rajesh,rajesh@yahoo.com,MALE,01-01-1990,hyderabad
1009,Sophie,sophie@gmail,F,1991/02/02,mumbai
1010,Ali khan,ali.khan@aol.com,male,not available,lahore


In [0]:
customers_clean = (
    customer
    .withColumn("email", lower(trim(col("EMAIL"))))
    .withColumn("name", initcap(trim(col("name"))))
    .withColumn("gender", when(lower(col("gender")).isin("f", "female"), "Female")
                          .when(lower(col("gender")).isin("m", "male"), "Male")
                          .otherwise("Other"))
    .withColumn("dob", coalesce(
        try_to_date(col("dob"), "yyyy-MM-dd"),
        try_to_date(col("dob"), "yyyy/MM/dd"),
        try_to_date(col("dob"), "dd-MM-yyyy"),
        try_to_date(col("dob"), "MM-dd-yyyy")
    ))
    .withColumn("location", initcap(col("location")))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id", "email"])
)

In [0]:
display(customers_clean)

customer_id,name,email,gender,dob,location
1001,John Doe,john.doe@gmail.com,Male,1990-05-12,New York
1002,Sara Khan,sara.khan@gmail.com,Female,1988-06-12,Mumbai
1003,null,tom@outlook.com,Male,null,San Francisco
1004,Anil Sharma,anil.sharma@yahoo.in,Male,1989-03-17,Bangalore
1005,Priya R,priya.r@hotmail.com,Female,1993-11-23,Delhi
1006,Amit Patel,amit123@gmail.com,Male,1992-08-19,Surat
1007,Meena Kumari,meena@rediffmail.com,Female,1995-04-10,Chennai
1008,Rajesh,rajesh@yahoo.com,Male,1990-01-01,Hyderabad
1009,Sophie,sophie@gmail,Female,1991-02-02,Mumbai
1010,Ali Khan,ali.khan@aol.com,Male,null,Lahore


In [0]:
customers_clean.write.format("delta").mode("overwrite").saveAsTable("fabric_lh.silver.customers")

In [0]:
orders=spark.read.table("fabric_lh.bronze.orders")
display(orders)

order_id,customer_id,order_date,amount,status
2001,1001,2022/07/15,120.5,Complete
2002,1002,15-07-2022,null,Pending
2003,1003,2022-07-18,250.0,shipped
2004,1004,20220719,0.0,complete
2005,1005,2022-07-20,-45.0,failed
2006,1006,2022/07/21,500.0,Complete
2007,1007,21-07-2022,300.0,shipped
2008,1008,20220722,100.75,pending
2009,1009,22-07-2022,200.0,Cancelled
2010,1010,2022/07/23,null,complete


In [0]:

orders_clean = (
    orders
    .withColumn
    (
        "order_date", coalesce
        (
        try_to_date(col("order_date"), "yyyy-MM-dd"),
        try_to_date(col("order_date"), "yyyy/MM/dd"),
        try_to_date(col("order_date"), "dd-MM-yyyy"),
        try_to_date(col("order_date"), "MM-dd-yyyy")
        )
    )
    .withColumn("order_id", col("order_id").cast("int"))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .withColumn("status", initcap(col("status")))
    .dropna(subset=["customer_id", "order_date"])
    .dropDuplicates(["order_id"])
)
display(orders_clean)

order_id,customer_id,order_date,amount,status
2001,1001,2022-07-15,120.5,Complete
2002,1002,2022-07-15,null,Pending
2003,1003,2022-07-18,250.0,Shipped
2005,1005,2022-07-20,null,Failed
2006,1006,2022-07-21,500.0,Complete
2007,1007,2022-07-21,300.0,Shipped
2009,1009,2022-07-22,200.0,Cancelled
2010,1010,2022-07-23,null,Complete
2013,1013,2022-07-24,180.0,Complete
2014,1014,2022-07-25,90.0,Shipped


In [0]:
orders_clean.write.format("delta").mode("overwrite").saveAsTable("  fabric_lh.silver.orders")

In [0]:
payments=spark.read.table("fabric_lh.bronze.payments")
display(payments)


payment_id,customer_id,payment_date,payment_method,payment_status,amount
P001,1001,2022-07-16,Credit Card,SUCCESS,120.5
P002,1002,2022/07/17,creditcard,Failed,null
P003,1003,17-07-2022,PayPal,Success,250.0
P004,1004,20220718,cash,success,-15.0
P005,1005,n/a,UPI,null,145.0
P006,1006,2022/07/19,Wallet,SUCCESS,500.0
P007,1007,2022-07-20,Credit Card,SUCCESS,300.0
P008,1008,21-07-2022,PayPal,Failed,0.0
P009,1009,20220722,Credit Card,Success,200.0
P010,1010,2022-07-23,UPI,success,100.0


In [0]:
payments_clean = (
    payments
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .withColumn("payment_date", coalesce(
        try_to_date(col("payment_date"), "yyyy-MM-dd"),
        try_to_date(col("payment_date"), "yyyy/MM/dd"),
        try_to_date(col("payment_date"), "dd-MM-yyyy"),
        try_to_date(col("payment_date"), "MM-dd-yyyy")
    ))
    .withColumn("payment_method", initcap(trim(col("payment_method"))))
    .withColumn("payment_status", initcap(trim(col("payment_status"))))
    .dropna(subset=["customer_id","payment_date", "amount"])
    .dropDuplicates(["payment_id"])
)
display(payments_clean)

payment_id,customer_id,payment_date,payment_method,payment_status,amount
P001,1001,2022-07-16,Credit Card,Success,120.5
P003,1003,2022-07-17,Paypal,Success,250.0
P006,1006,2022-07-19,Wallet,Success,500.0
P007,1007,2022-07-20,Credit Card,Success,300.0
P008,1008,2022-07-21,Paypal,Failed,0.0
P010,1010,2022-07-23,Upi,Success,100.0
P012,1012,2022-07-24,Cash,Success,150.5
P013,1013,2022-07-24,Credit Card,Success,180.0
P014,1014,2022-07-25,Upi,Failed,90.0
P015,1015,2022-07-26,Netbanking,Success,0.0


In [0]:
payments_clean.write.format("delta").mode("overwrite").saveAsTable("fabric_lh.silver.payments")

In [0]:
support=spark.read.table("fabric_lh.bronze.support_tickets")
display(support)


ticket_id,customer_id,issue_type,ticket_date,resolution_status
T001,1001,Payment Issue,2022-07-19,closed
T002,1002,delivery delay,19-07-2022,pending
T003,1003,PAYMENT ISSUE,20220720,Closed
T004,1004,login problem,2022/07/20,null
T005,1005,NA,n/a,resolved
T006,1006,refund request,2022/07/21,Pending
T007,1007,Delivery Delay,21-07-2022,Closed
T008,1008,Product damaged,20220722,Resolved
T009,1009,login issue,2022/07/23,pending
T010,1010,Support needed,2022-07-23,null


In [0]:
support_clean = (
    support
        .withColumn("issue_type", initcap(trim(col("issue_type"))))
        .withColumn("ticket_date", 
                    coalesce(
                            try_to_date(col("ticket_date"), "yyyy-MM-dd"),
                            try_to_date(col("ticket_date"), "yyyy/MM/dd"),
                            try_to_date(col("ticket_date"), "dd-MM-yyyy"),
                            try_to_date(col("ticket_date"), "MM-dd-yyyy"),
                            try_to_date(col("ticket_date"), "yyyyMMdd")
                            )
                    )
        .withColumn("resolution_status",initcap(trim(col("resolution_status"))))
        .dropna(subset=["customer_id","ticket_date"])
        .dropDuplicates(["ticket_id"])
)
display(support_clean)

ticket_id,customer_id,issue_type,ticket_date,resolution_status
T001,1001,Payment Issue,2022-07-19,Closed
T002,1002,Delivery Delay,2022-07-19,Pending
T003,1003,Payment Issue,2022-07-20,Closed
T004,1004,Login Problem,2022-07-20,null
T006,1006,Refund Request,2022-07-21,Pending
T007,1007,Delivery Delay,2022-07-21,Closed
T008,1008,Product Damaged,2022-07-22,Resolved
T009,1009,Login Issue,2022-07-23,Pending
T010,1010,Support Needed,2022-07-23,null
T012,1012,Na,2022-07-24,Resolved


In [0]:
support_clean.write.format("delta").mode("overwrite").saveAsTable("fabric_lh.silver.support_trickets")

In [0]:
web=spark.read.table("fabric_lh.bronze.web_activities")
display(web)

session_id,customer_id,page_viewed,session_time,device_type
S001,1001,/Home,2022-07-18,android
S002,1002,/products/shoes,18-07-2022,IOS
S003,1003,/PRODUCTS/BAGS,20220719,Windows
S004,1004,/cart,2022-07-19,linux
S005,1005,/checkout,2022/07/20,android
S006,1006,/Home,20-07-2022,iOS
S007,1007,/Products/Electronics,20220721,Windows
S008,1008,/products/shoes,2022-07-22,macOS
S009,1009,/home,22/07/2022,iOS
S010,1010,/home,2022/07/23,android


In [0]:
web_clean = (
    web
    .withColumn("session_time", 
                coalesce(
                            try_to_date(col("session_time"), "yyyy-MM-dd"),
                            try_to_date(col("session_time"), "yyyy/MM/dd"),
                            try_to_date(col("session_time"), "dd-MM-yyyy"),
                            try_to_date(col("session_time"), "MM-dd-yyyy"),
                            try_to_date(col("session_time"), "yyyyMMdd")
                            ))
    .withColumn("page_viewed", lower(col("page_viewed")))
    .withColumn("device_type", initcap(col("device_type")))
    .dropDuplicates(["session_id"])
    .dropna(subset=["customer_id", "session_time", "page_viewed"])
)
display(web_clean)


session_id,customer_id,page_viewed,session_time,device_type
S001,1001,/home,2022-07-18,Android
S002,1002,/products/shoes,2022-07-18,Ios
S003,1003,/products/bags,2022-07-19,Windows
S004,1004,/cart,2022-07-19,Linux
S005,1005,/checkout,2022-07-20,Android
S006,1006,/home,2022-07-20,Ios
S007,1007,/products/electronics,2022-07-21,Windows
S008,1008,/products/shoes,2022-07-22,Macos
S010,1010,/home,2022-07-23,Android
S011,1011,/products/shoes,2022-07-23,Ios


In [0]:
web_clean.write.format("delta").mode("overwrite").saveAsTable("fabric_lh.silver.web_activities")